# CGAP statistical significance testing — full analysis
Use this after the **3-seed × 15-participant LOSO** CGAP ablation has completed. It reads the modelling notebook's checkpoint CSV; it does not fabricate or regenerate predictions.

**Independent unit:** participant. Seed results are averaged within each held-out participant before inferential tests, avoiding pseudo-replication from treating 45 participant-seed results as independent.

Includes participant-level descriptive statistics, Friedman omnibus test, paired Wilcoxon tests (CGAP Full vs each other variant), Holm correction, paired bootstrap 95% CIs, matched-pairs rank-biserial effect sizes, and a sign-flip permutation sensitivity analysis.

In [ ]:
# 1. Setup
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

# Change this if your completed CGAP checkpoint CSV is stored elsewhere.
RESULTS_FILE = Path('/content/drive/MyDrive/DataCon/Project 6 Wesad/cgap_ablation_checkpoints/ablation_results_checkpoint.csv')
OUTPUT_DIR = RESULTS_FILE.parent / 'statistical_significance'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_BOOT = 20000
N_PERM = 50000
ALPHA = 0.05
REFERENCE_MODEL = 'CGAP Full'
MODEL_ORDER = ['CGAP Full', 'Global only', 'Online only',
               'CGAP no online updates', 'CGAP no class weights']
METRIC_ALIASES = {
    'accuracy': ['accuracy', 'acc'],
    'balanced_accuracy': ['balanced_accuracy', 'balanced_acc'],
    'macro_f1': ['macro_f1', 'f1_macro', 'macro-f1'],
    'macro_precision': ['macro_precision', 'precision_macro'],
    'macro_recall': ['macro_recall', 'recall_macro'],
}
assert RESULTS_FILE.exists(), f'CSV not found: {RESULTS_FILE}. Run the LOSO notebook first or update RESULTS_FILE.'
raw = pd.read_csv(RESULTS_FILE)
print('Input shape:', raw.shape)
print('Columns:', raw.columns.tolist())
display(raw.head())

In [ ]:
# 2. Detect and standardise columns
def norm(s):
    return ''.join(c.lower() for c in str(s) if c.isalnum())

def find_col(columns, aliases, required=True):
    lookup = {norm(c): c for c in columns}
    for a in aliases:
        if norm(a) in lookup:
            return lookup[norm(a)]
    if required:
        raise ValueError(f'Cannot find column matching {aliases}. Available: {list(columns)}')
    return None

model_col = find_col(raw.columns, ['model', 'variant', 'model_name'])
seed_col = find_col(raw.columns, ['seed', 'random_seed'])
subject_col = find_col(raw.columns, ['held_out_subject', 'subject', 'heldout_subject',
                                     'test_subject', 'participant', 'held_out'], required=False)
fold_col = find_col(raw.columns, ['fold', 'outer_fold'], required=False)
metric_cols = {m: find_col(raw.columns, aliases, required=False)
               for m, aliases in METRIC_ALIASES.items()}
metric_cols = {m:c for m,c in metric_cols.items() if c is not None}
if not metric_cols:
    raise ValueError('No metric columns recognised. Update METRIC_ALIASES to match the CSV.')

rename = {model_col:'model', seed_col:'seed'}
if subject_col: rename[subject_col] = 'subject'
if fold_col: rename[fold_col] = 'fold'
rename.update({c:m for m,c in metric_cols.items()})
df = raw.rename(columns=rename).copy()

# If subject ID is absent, do not silently assume folds are subjects unless one-to-one mapping is verified.
if 'subject' not in df.columns:
    if 'fold' not in df.columns:
        raise ValueError('Need held-out participant column or a fold column mapping one-to-one to participants.')
    map_check = df.groupby('fold')['model'].nunique()
    print('No subject column. Fold mapping diagnostic:', map_check.to_dict())
    df['subject'] = df['fold'].astype(str)
    warnings.warn('Using fold as participant identifier. Verify each LOSO fold maps to exactly one unique participant.')

df['model'] = df['model'].astype(str).str.strip()
df['subject'] = df['subject'].astype(str).str.strip()
df['seed'] = pd.to_numeric(df['seed'], errors='raise').astype(int)
metrics = list(metric_cols.keys())
for m in metrics:
    df[m] = pd.to_numeric(df[m], errors='coerce')
df = df.dropna(subset=['model','subject','seed'] + metrics)

print('Detected metrics:', metrics)
print('Models:', sorted(df.model.unique()))
print('Seeds:', sorted(df.seed.unique()))
print('Subjects:', sorted(df.subject.unique()))
display(pd.DataFrame({'metric':list(metric_cols), 'source_column':list(metric_cols.values())}))

In [ ]:
# 3. Validate uniqueness and coverage
key = ['seed','subject','model']
dups = df[df.duplicated(key, keep=False)]
if not dups.empty:
    display(dups.sort_values(key))
    raise ValueError('Duplicate seed × subject × model rows. Resolve before significance tests.')

coverage = df.groupby(['model','subject'])['seed'].nunique().unstack(fill_value=0)
print('Seed coverage (normally 3 for every model and subject):')
display(coverage)
if (coverage != 3).any().any():
    warnings.warn('Incomplete seed coverage detected. The paired analysis below will keep only complete participant blocks.')

models = [m for m in MODEL_ORDER if m in df.model.unique()]
models += [m for m in sorted(df.model.unique()) if m not in models]
print('Models included:', models)

# Average across seeds within each participant × model; participant is the inferential unit.
participant = df.groupby(['subject','model'], as_index=False)[metrics].mean()
participant.to_csv(OUTPUT_DIR / 'participant_means_across_seeds.csv', index=False)

def wide_metric(metric):
    w = participant.pivot(index='subject', columns='model', values=metric)
    use = [m for m in models if m in w.columns]
    return w[use].dropna(axis=0, how='any')

desc = []
for m in models:
    for metric in metrics:
        vals = participant.loc[participant.model.eq(m), metric].dropna()
        desc.append({'model':m,'metric':metric,'n_participants':len(vals),
                     'mean':vals.mean(),'sd':vals.std(ddof=1),'median':vals.median(),
                     'min':vals.min(),'max':vals.max()})
descriptive = pd.DataFrame(desc)
display(descriptive.round(4))
descriptive.to_csv(OUTPUT_DIR / 'descriptive_statistics.csv', index=False)

In [ ]:
# 4. Helpers: paired bootstrap CI, rank-biserial effect size, robust Wilcoxon
def bootstrap_ci(d, n_boot=N_BOOT, seed=12345, conf=0.95):
    d = np.asarray(d, float)
    d = d[np.isfinite(d)]
    if not len(d): return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    # Resample participants' paired differences.
    means = np.empty(n_boot)
    for start in range(0, n_boot, 1000):
        n = min(1000, n_boot-start)
        means[start:start+n] = rng.choice(d, size=(n,len(d)), replace=True).mean(axis=1)
    q = (1-conf)/2
    return float(np.quantile(means,q)), float(np.quantile(means,1-q))

def rank_biserial(d):
    d = np.asarray(d,float)
    d = d[np.isfinite(d) & (np.asarray(d) != 0)]
    if not len(d): return 0.0
    ranks = pd.Series(np.abs(d)).rank(method='average').to_numpy()
    plus, minus = ranks[d>0].sum(), ranks[d<0].sum()
    return float((plus-minus)/(plus+minus)) if plus+minus else 0.0

def paired_wilcoxon(a,b):
    d = np.asarray(a,float)-np.asarray(b,float)
    if len(d)<2 or np.allclose(d,0): return 0.0,1.0,d
    try:
        r = wilcoxon(d, alternative='two-sided', zero_method='wilcox', method='auto')
    except TypeError:
        r = wilcoxon(d, alternative='two-sided', zero_method='wilcox', mode='auto')
    return float(r.statistic),float(r.pvalue),d

In [ ]:
# 5. Friedman omnibus test across model variants
friedman_rows = []
for metric in metrics:
    w = wide_metric(metric)
    if w.shape[1] < 3 or w.shape[0] < 3:
        warnings.warn(f'Skipping Friedman for {metric}: insufficient complete models/participants.')
        continue
    stat,p = friedmanchisquare(*[w[m].to_numpy() for m in w.columns])
    n,k = w.shape
    # Kendall's W effect size for Friedman.
    friedman_rows.append({'metric':metric,'n_participants':n,'n_models':k,
                          'friedman_chi2':float(stat),'p_value':float(p),
                          'kendalls_W':float(stat/(n*(k-1)))})
friedman = pd.DataFrame(friedman_rows)
display(friedman.round(6))
friedman.to_csv(OUTPUT_DIR / 'friedman_omnibus.csv', index=False)

In [ ]:
# 6. Planned post-hoc comparisons: CGAP Full vs each other model
# Holm correction is applied across the comparator family separately for each metric.
posthoc_rows = []
for metric in metrics:
    w = wide_metric(metric)
    if REFERENCE_MODEL not in w.columns:
        warnings.warn(f'{REFERENCE_MODEL} missing for {metric}; skipping.')
        continue
    ref = w[REFERENCE_MODEL].to_numpy()
    rows = []
    for comp_model in [m for m in w.columns if m != REFERENCE_MODEL]:
        comp = w[comp_model].to_numpy()
        stat,p,d = paired_wilcoxon(ref,comp)
        lo,hi = bootstrap_ci(d)
        rows.append({'metric':metric,'reference_model':REFERENCE_MODEL,
                     'comparison_model':comp_model,'n_participants':len(d),
                     'reference_mean':float(ref.mean()),'comparison_mean':float(comp.mean()),
                     'mean_difference_CGAP_minus_comparator':float(d.mean()),
                     'median_difference':float(np.median(d)),
                     'bootstrap_95ci_low':lo,'bootstrap_95ci_high':hi,
                     'wilcoxon_statistic':stat,'p_raw':p,
                     'rank_biserial_effect':rank_biserial(d)})
    if rows:
        reject,padj,_,_ = multipletests([r['p_raw'] for r in rows], alpha=ALPHA, method='holm')
        for r,pa,re in zip(rows,padj,reject):
            r['p_holm'] = float(pa)
            r['significant_holm_0_05'] = bool(re)
            posthoc_rows.append(r)
posthoc = pd.DataFrame(posthoc_rows)
display(posthoc.round(6))
posthoc.to_csv(OUTPUT_DIR / 'wilcoxon_posthoc_holm.csv', index=False)

In [ ]:
# 7. Sign-flip permutation sensitivity test on paired participant differences
def sign_flip_p(d, n_perm=N_PERM, seed=2026):
    d = np.asarray(d,float)
    d = d[np.isfinite(d)]
    if not len(d): return np.nan
    observed = abs(d.mean())
    rng = np.random.default_rng(seed)
    exceed = 0
    done = 0
    while done < n_perm:
        b = min(5000,n_perm-done)
        signs = rng.choice([-1.0,1.0],size=(b,len(d)))
        exceed += int((np.abs((signs*d).mean(axis=1)) >= observed).sum())
        done += b
    return (exceed+1)/(n_perm+1)

perm_rows=[]
for metric in metrics:
    w=wide_metric(metric)
    if REFERENCE_MODEL not in w.columns: continue
    for comp_model in [m for m in w.columns if m != REFERENCE_MODEL]:
        d=w[REFERENCE_MODEL].to_numpy()-w[comp_model].to_numpy()
        perm_rows.append({'metric':metric,'reference_model':REFERENCE_MODEL,
                          'comparison_model':comp_model,'n_participants':len(d),
                          'mean_difference':float(d.mean()),
                          'sign_flip_p_raw':float(sign_flip_p(d))})
perm=pd.DataFrame(perm_rows)
display(perm.round(6))
perm.to_csv(OUTPUT_DIR/'sign_flip_permutation_supplementary.csv',index=False)

In [ ]:
# 8. Save paired values and a concise audit report
for metric in metrics:
    wide_metric(metric).to_csv(OUTPUT_DIR / f'paired_values_{metric}.csv')

lines = [
    'CGAP Statistical Significance Analysis',
    f'Input CSV: {RESULTS_FILE}',
    f'Input rows: {len(raw)}',
    f'Participants represented: {df.subject.nunique()}',
    f'Seeds represented: {sorted(df.seed.unique().tolist())}',
    f'Models: {models}',
    '',
    'Analysis unit: participant. Seeds averaged within participant before testing.',
    'Friedman is the omnibus test; Wilcoxon tests compare CGAP Full with each comparator.',
    'Holm adjustment is applied separately for each metric across its post-hoc comparisons.',
    'Bootstrap confidence intervals are for paired participant-level mean differences.',
    'Positive CGAP-minus-comparator differences favor CGAP when higher metric values are better.',
    'Do not interpret missing/incomplete participant coverage as a completed 15-subject experiment.',
    '',
    'Output files:'
] + [f'- {p.name}' for p in sorted(OUTPUT_DIR.glob('*.csv'))]
(OUTPUT_DIR/'README_analysis.txt').write_text('\\n'.join(lines),encoding='utf-8')
print('\\n'.join(lines))
print('All outputs saved to:', OUTPUT_DIR)

## How to report the tests
Report mean ± SD across the 15 participant-level values, then report Friedman χ², exact p-value and Kendall's W. For planned CGAP-versus-ablation comparisons, report the paired Wilcoxon p-value after Holm correction, the paired bootstrap 95% CI for the mean difference, and the rank-biserial effect size.

Do not claim significance until the full LOSO results are complete and participant coverage is verified. Do not treat windows or repeated seeds as independent participants. If only aggregated metric CSVs are available, window-level McNemar testing is not supported; that requires paired per-window predictions.